In [32]:
import pandas as pd
import numpy as np

df = pd.read_csv('adult.data', sep = ',', header = None, skipinitialspace=True)
features = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
'marital-status', 'occupation', 'relationship', 'race', 'sex',
'capital-gain', 'capital-loss', 'hours-per-week',
'native-country', 'salary']
df.columns = features
df.replace('?', np.NaN, inplace=True)
df.dropna(inplace= True)
df.drop_duplicates(inplace=True)
df = df.sample(frac=1)
df.reset_index(drop=True, inplace=True)
df_num = pd.get_dummies(df, columns = ['workclass', 'education', 'marital-status',
'occupation', 'relationship', 'race', 'sex', 'native-country'])
df_num['salary'].replace(['>50K', '<=50K'], [1,-1], inplace=True)



In [33]:
print(df.head())

   age         workclass  fnlwgt     education  education-num  \
0   59           Private   43221           9th              5   
1   53  Self-emp-not-inc   98829     Bachelors             13   
2   45      Self-emp-inc  281911       HS-grad              9   
3   36           Private  219483  Some-college             10   
4   29           Private  188675  Some-college             10   

       marital-status        occupation   relationship   race     sex  \
0  Married-civ-spouse  Transport-moving        Husband  White    Male   
1  Married-civ-spouse    Prof-specialty        Husband  White    Male   
2  Married-civ-spouse      Craft-repair        Husband  White    Male   
3       Never-married      Adm-clerical  Not-in-family  White  Female   
4  Married-civ-spouse      Craft-repair        Husband  Black    Male   

   capital-gain  capital-loss  hours-per-week native-country salary  
0             0             0              60  United-States   >50K  
1             0             0 

In [31]:
print(df_num.head())

   age  fnlwgt  education-num  capital-gain  capital-loss  hours-per-week  \
0   47  386136             12             0             0              40   
1   49  146268             10             0             0              40   
2   22   99697              9             0             0              40   
3   23  117363             13             0             0              40   
4   30  182714             10             0             0              40   

   salary  workclass_Federal-gov  workclass_Local-gov  workclass_Private  ...  \
0       1                      0                    0                  1  ...   
1      -1                      0                    0                  1  ...   
2      -1                      0                    0                  1  ...   
3      -1                      0                    0                  1  ...   
4      -1                      0                    0                  1  ...   

   native-country_Portugal  native-country_Puerto-

In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('adult.data', sep=',', header=None, skipinitialspace=True)
features = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
            'marital-status', 'occupation', 'relationship', 'race', 'sex',
            'capital-gain', 'capital-loss', 'hours-per-week',
            'native-country', 'salary']
df.columns = features
df.replace('?', np.NaN, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df = df.sample(frac=1).reset_index(drop=True)

df_num = pd.get_dummies(df, columns=['workclass', 'education', 'marital-status',
                                     'occupation', 'relationship', 'race', 'sex', 'native-country'])
df_num['salary'].replace(['>50K', '<=50K'], [1, -1], inplace=True)

X = df_num.drop(columns=['salary'])
y = df_num['salary']

scaler = StandardScaler()
X = scaler.fit_transform(X)


S = [] 
F = list(range(X.shape[1]))
best_score = 0 
k = 3  

# Forward feature selection
while True:
    improved = False
    best_feature = None
    for feature in F:
        if feature not in S:
            temp_features = S + [feature]
            X_temp = X[:, temp_features]
            
            X_train, X_test, y_train, y_test = train_test_split(X_temp, y, test_size=0.3, random_state=42)
            
            knn = KNeighborsClassifier(n_neighbors=k)
            knn.fit(X_train, y_train)
            y_pred = knn.predict(X_test)
            
            # Calculate balanced accuracy
            score = balanced_accuracy_score(y_test, y_pred)
            
            if score > best_score:
                best_score = score
                best_feature = feature
                improved = True
    
    
    if improved and best_feature is not None:
        S.append(best_feature)
        print(f"Added feature {best_feature}, new best score: {best_score}")
    else:
       
        break

X_final = X[:, S]
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.3, random_state=40)
knn_final = KNeighborsClassifier(n_neighbors=k)
knn_final.fit(X_train, y_train)
y_pred_final = knn_final.predict(X_test)

# Report results
final_score = balanced_accuracy_score(y_test, y_pred_final)
print("\nSelected features:", S)
print("Final Balanced Accuracy:", final_score)
print("Accuracy for majority class:", accuracy_score(y_test[y_test == 1], y_pred_final[y_test == 1]))
print("Accuracy for minority class:", accuracy_score(y_test[y_test == -1], y_pred_final[y_test == -1]))


Added feature 50, new best score: 0.7208569483372974
Added feature 3, new best score: 0.7569920551414641
Added feature 4, new best score: 0.7703131278299936
Added feature 19, new best score: 0.774884147713262
Added feature 31, new best score: 0.794449503925273

Selected features: [50, 3, 4, 19, 31]
Final Balanced Accuracy: 0.778239164315365
Accuracy for majority class: 0.8223920863309353
Accuracy for minority class: 0.7340862422997947
